# Level 1: Geometric Intuition, Cochains, and Why Difference is Not Repair

**Paper Package 002** — Pedagogical Progression: Level 1 of 5

**Target Audience:** Beginners with basic programming experience. No background in differential geometry, quantum field theory, or general relativity is assumed.

**Core Idea:** In physical law and mathematics, observing a discrepancy (a *raw difference*) is not the same as finding an unfixable defect (an *obstruction*). An obstruction is a discrepancy that cannot be removed by any valid internal adjustment (a *gauge transformation*).

*Evidence scope: finite toy-model demonstration only.*

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Locate and import code directory dynamically
cwd = Path.cwd().resolve()
for candidate in (cwd, cwd / "code", cwd.parent / "code", *(d / "code" for d in cwd.parents)):
    if (candidate / "constraint_closure_l1_structure.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from poset import Poset
print('[OK] Environment initialized successfully!')

## 1. What is a Discrete Cochain?

Imagine a simple network of points (vertices) connected by directional lines (edges):
- A **0-cochain** $s$ assigns a single real number to each **vertex** (like temperature or height at each station).
- A **1-cochain** $\omega$ assigns a real number to each **directed edge** (like water flow or force along a pipe).

The **coboundary operator** $d_0$ takes a vertex assignment $s$ and measures the difference across each edge:
$$\delta(e) = (d_0 s)(e) = s(\text{head}) - s(\text{tail}).$$

Let's test this on a single directed edge $0 \to 1$:

In [ ]:
# A single edge connecting vertex 0 to vertex 1
single_edge_poset = Poset(num_vertices=2, edges=[(0, 1)])

# Suppose vertex 0 has value 1.0, and vertex 1 has value 0.0
s = np.array([1.0, 0.0])
delta_s = single_edge_poset.coboundary(s)

print(f"Vertex values s:       {s}")
print(f"Edge difference d0*s:  {delta_s}")
raw_variation = float(np.sum(np.abs(delta_s)))
print(f"Raw total variation:   {raw_variation}")

# Can we eliminate this edge difference by adjusting vertex values?
# Yes! Shift vertex 1 to 1.0 (or vertex 0 to 0.0):
s_repaired = np.array([1.0, 1.0])
delta_repaired = single_edge_poset.coboundary(s_repaired)
repaired_variation = float(np.sum(np.abs(delta_repaired)))
print(f"Repaired difference:   {delta_repaired} (Variation: {repaired_variation})")

assert raw_variation == 1.0
assert repaired_variation == 0.0
print("[OK] Single edge discrepancy was completely repairable!")

## 2. The 3-Cycle (Triangle) Obstruction

Now consider three vertices arranged in a directed loop (a 3-cycle $C_3$):
- Edge 0: $0 \to 1$
- Edge 1: $1 \to 2$
- Edge 2: $2 \to 0$

For **any** choice of vertex values $s = (s_0, s_1, s_2)$,
the edge differences are:
$$\delta_0 = s_1 - s_0, \quad \delta_1 = s_2 - s_1, \quad \delta_2 = s_0 - s_2.$$

Notice what happens when you sum the differences all the way around the loop:
$$\sum_{i=0}^2 \delta_i = (s_1 - s_0) + (s_2 - s_1) + (s_0 - s_2) = 0.$$

**The loop sum of any exact difference is ALWAYS zero!**

What if someone gives us edge measurements that have a nonzero circulation sum, say $\omega = (1.0, 1.0, 1.0)$?

In [ ]:
cycle_poset = Poset(num_vertices=3, edges=[(0, 1), (1, 2), (2, 0)])

# An edge cochain with net circulation: sum(omega) = 1 + 1 + 1 = 3
omega = np.array([1.0, 1.0, 1.0])
circulation = float(np.sum(omega))
print(f"Cochain on edges:      {omega}")
print(f"Total loop circulation:{circulation}")

# In the quotient space C^1 / im(d0), the irreducible residual is Phi_1([omega]) = |sum(omega)| = 3.0
# No choice of vertex shifts s can reduce this residual below 3.0!
from regge_susy_saturation_solver import compute_primitive_difference_quotient

res = compute_primitive_difference_quotient(omega, cycle_poset.d0)
min_residual = res["min_l1"]
print(f"Optimal L1 repair cost Phi_1: {min_residual:.6f}")

assert np.isclose(min_residual, 3.0)
print("[OK] Verified: The circulation is an irreducible topological obstruction!")

## 3. Summary: Difference vs. Obstruction

| Concept | Mathematical Symbol | Physical Meaning |
| :--- | :--- | :--- |
| Raw Difference | $\delta = d_0 s$ | Local gradient; can be calibrated away by vertex gauge shifts. |
| Total Circulation | $\sum_{e \in \text{loop}} \omega(e)$ | Global holonomy around a cycle; cannot be removed by local shifts. |
| Quotient Residual | $\Phi_1([\omega]) = \inf_\alpha \|\omega - d_0 \alpha\|_1$ | The minimal remaining discrepancy after applying optimal gauge repairs. |

In **Level 2**, we will explore simplicial complexes, 2D triangular faces, and why the choice of norm ($\ell^1$ vs $\ell^2$) fundamentally determines physical behavior.